# NutriGo · Week 2 — Data collection & exploration (EDA)

Inputs
- `data/food101_nutrition.csv` — 101 Food-101 dishes (per serving)
- `data/healthy_alternatives.csv` — 50 healthier swaps
- `fastfood.csv` — Kaggle *Fastfood Nutrition* (515 items, 8 chains, 17 columns).
  On Kaggle add it as an input; locally download it into `data/fastfood.csv`.

Charts are saved to `reports/figures/`.

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

ROOT = '..' if os.path.exists('../data') else '.'
DATA = os.path.join(ROOT, 'data')
FIG = os.path.join(ROOT, 'reports', 'figures'); os.makedirs(FIG, exist_ok=True)

foods = pd.read_csv(os.path.join(DATA, 'food101_nutrition.csv'))
alts = pd.read_csv(os.path.join(DATA, 'healthy_alternatives.csv'))

def health_score(df):
    s = 60 - (df.kcal/40 + df.sat_fat_g + 0.4*df.sugar_g + df.sodium_mg/150) + (0.5*df.protein_g + 1.5*df.fibre_g)
    return s.clip(0, 100).round().astype(int)

foods['health_score'] = health_score(foods)
alts['health_score'] = health_score(alts)
print(foods.shape, alts.shape)
foods.head()

## 1 · Data quality checks

In [ ]:
num = ['kcal','protein_g','carbs_g','fat_g','sat_fat_g','fibre_g','sugar_g','sodium_mg']
print('missing values:', int(foods.isna().sum().sum() + alts.isna().sum().sum()))
print('duplicate ids :', foods.food_id.duplicated().sum(), alts.food_id.duplicated().sum())
# macros must add up to kcal (4/4/9)
calc = 4*foods.protein_g + 4*foods.carbs_g + 9*foods.fat_g
print('max kcal mismatch:', float((calc - foods.kcal).abs().max()))
foods[num].describe().round(1)

## 2 · Where the calories are: average kcal per serving by category

In [ ]:
cat = foods.groupby('category').kcal.mean().sort_values()
ax = cat.plot.barh(color='#2e7d4f', figsize=(8, 5))
for i, v in enumerate(cat):
    ax.text(v + 8, i, f'{v:.0f}', va='center')
ax.set_title('101 Food-101 dishes grouped into 12 categories'); ax.set_xlabel('kcal per serving')
plt.tight_layout(); plt.savefig(f'{FIG}/kcal_by_category.png', dpi=150); plt.show()
cat.sort_values(ascending=False).round()

## 3 · Swap catalog vs Food-101

In [ ]:
f_mean, a_mean = foods.kcal.mean(), alts.kcal.mean()
print(f'Food-101 average: {f_mean:.0f} kcal | swaps average: {a_mean:.0f} kcal | {100*(1-a_mean/f_mean):.0f}% lighter')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist([foods.kcal, alts.kcal], bins=15, label=['Food-101', 'Swaps'], color=['#b9772a', '#2e7d4f'])
axes[0].set_title('kcal per serving'); axes[0].legend()
axes[1].hist([foods.health_score, alts.health_score], bins=15, label=['Food-101', 'Swaps'], color=['#b9772a', '#2e7d4f'])
axes[1].set_title('Health score (0-100)'); axes[1].legend()
plt.tight_layout(); plt.savefig(f'{FIG}/foods_vs_swaps.png', dpi=150); plt.show()

## 4 · Nutrients that drive the health score

In [ ]:
corr = foods[num + ['health_score']].corr()['health_score'].drop('health_score').sort_values()
corr.plot.barh(color=['#c0392b' if v < 0 else '#2e7d4f' for v in corr], figsize=(7, 4), title='Correlation with health score')
plt.tight_layout(); plt.savefig(f'{FIG}/health_score_drivers.png', dpi=150); plt.show()
foods.nsmallest(10, 'health_score')[['name', 'category', 'kcal', 'sat_fat_g', 'sodium_mg', 'health_score']]

## 5 · Kaggle Fastfood Nutrition (real menu items)

In [ ]:
paths = glob.glob('/kaggle/input/**/fastfood.csv', recursive=True) + glob.glob(os.path.join(DATA, 'fastfood.csv'))
if not paths:
    print('fastfood.csv not found - add the Kaggle dataset "ulrikthygepedersen/fastfood-nutrition"')
    ff = None
else:
    ff = pd.read_csv(paths[0])
    print(ff.shape, '| chains:', ff.restaurant.nunique())
    display(ff.head())

In [ ]:
if ff is not None:
    by_chain = ff.groupby('restaurant').calories.agg(['count', 'mean', 'max']).sort_values('mean')
    by_chain['mean'].plot.barh(color='#2e7d4f', figsize=(8, 4), title='Average kcal per menu item by chain')
    plt.tight_layout(); plt.savefig(f'{FIG}/fastfood_kcal_by_chain.png', dpi=150); plt.show()
    display(by_chain.round())

In [ ]:
if ff is not None:
    grilled = ff[ff['item'].str.contains('grilled', case=False)]
    crispy = ff[ff['item'].str.contains('crispy|fried', case=False)]
    print(f'grilled items: {grilled.calories.mean():.0f} kcal avg ({len(grilled)}) | crispy/fried: {crispy.calories.mean():.0f} kcal avg ({len(crispy)})')
    display(ff.nlargest(10, 'calories')[['restaurant', 'item', 'calories', 'sodium']])

## Findings (fill in)
- Heaviest groups: meat, fast food, pasta and rice dishes average 600+ kcal.
- The 50 swaps average ~294 kcal vs ~450 for Food-101 (about 35% lighter).
- Sodium and saturated fat are the biggest negatives in the health score.
- Fast-food chains: grilled items are clearly lighter than crispy/fried ones → real menu swaps.